# TensorRT INT8 Optimization + Benchmark (Google Colab · GPU)

Fully automatic on **Run All**. Mirrors `src/optimization/*` and the patterns in `train_colab.ipynb`. Produces the **Optimization** row of the README results table (FP32 vs ONNX-INT8 vs TensorRT-INT8).

## Run steps
1. **Pick a GPU:** *Runtime → Change runtime type → T4 GPU* (free; sufficient — no A100 needed).
2. *Runtime → Run all*. Nothing else: this notebook needs **no DagsHub token** and no manual edits.

## Inputs (pulled automatically from Drive)
- `MyDrive/object-detection-tracking/weights/best.pt` — the fine-tuned model (from `train_colab.ipynb`).
- `MyDrive/object-detection-tracking/cache/visdrone_yolo.tar.gz` — the dataset cache (val images are used for INT8 calibration + the benchmark).

## Outputs (persisted back to Drive)
`best.onnx`, `best.int8.onnx`, `best.int8.engine`, and the benchmark markdown — copied to `MyDrive/object-detection-tracking/weights/`.

> **Note on INT8:** export + INT8 quantize are CPU steps; only the TensorRT build + GPU benchmark need the GPU (kept together here for one-click reproducibility). INT8's main win on CPU is **size reduction** — latency speedup can be modest without VNNI/AVX-512, while TensorRT INT8 on the GPU is where the speedup shows. **Report whatever the benchmark actually prints; do not fabricate numbers.**

In [ ]:
# --- Verify GPU (T4 is enough; any CUDA GPU works) ---
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU (T4).'
print('GPU:', torch.cuda.get_device_name(0))
!nvcc --version | tail -1

In [ ]:
# --- Install deps (export/quantize are CPU; TensorRT build/benchmark use GPU) ---
!pip -q install ultralytics onnx onnxruntime onnxruntime-gpu pycuda
# Colab usually ships a CUDA-compatible TensorRT; install a wheel only if missing.
try:
    import tensorrt as trt
    print('TensorRT', trt.__version__)
except ImportError:
    !pip -q install tensorrt
    import tensorrt as trt
    print('TensorRT', trt.__version__)

In [ ]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/object-detection-tracking'
print('Drive project root:', DRIVE_ROOT)

In [ ]:
# --- Resumable clone + install the repo (so src.optimization.* is importable) ---
import os
if not os.path.isdir('object-detection-tracking'):
    !git clone https://github.com/mohanemg07-web/object-detection-tracking.git
%cd object-detection-tracking
!pip -q install -r requirements.txt

In [ ]:
# --- Pull inputs from Drive automatically (fail clearly if missing) ---
import os, shutil, sys, tarfile

DRIVE_BEST = f'{DRIVE_ROOT}/weights/best.pt'
DATASET_CACHE = f'{DRIVE_ROOT}/cache/visdrone_yolo.tar.gz'
CALIB_DIR = '/content/data/yolo/VisDrone-DET/images/val'

# 1) best.pt -> local weights/best.pt
os.makedirs('weights', exist_ok=True)
if not os.path.exists(DRIVE_BEST):
    sys.exit(
        f'ERROR: trained model not found at {DRIVE_BEST}. '
        'Run train_colab.ipynb first so it saves best.pt to Drive.'
    )
shutil.copy2(DRIVE_BEST, 'weights/best.pt')
print('Copied best.pt ->', os.path.abspath('weights/best.pt'))

# 2) dataset cache -> /content/data/yolo (for calibration + benchmark)
if not os.path.exists(DATASET_CACHE):
    sys.exit(
        f'ERROR: dataset cache not found at {DATASET_CACHE}. '
        'Run train_colab.ipynb first so it creates the dataset tarball on Drive.'
    )
os.makedirs('/content/data/yolo', exist_ok=True)
with tarfile.open(DATASET_CACHE, 'r:gz') as tar:
    tar.extractall('/content/data/yolo')
assert os.path.isdir(CALIB_DIR), f'calibration images missing: {CALIB_DIR}'
n_calib = len([f for f in os.listdir(CALIB_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'Dataset cache extracted. Calibration images in {CALIB_DIR}: {n_calib}')

## Stage 2 chain: export → INT8 quantize → TensorRT engine → benchmark

Each step uses the repo's `src.optimization.*` CLIs with their exact flags.

In [ ]:
# --- (a) Export best.pt -> weights/best.onnx (with PyTorch<->ONNX parity check) ---
!python -m src.optimization.export_onnx --weights weights/best.pt --imgsz 640
# Expect a line: [parity] max abs diff = ... -> PASS

In [ ]:
# --- (b) Static INT8 quantization: weights/best.onnx -> weights/best.int8.onnx ---
# Calibrates on the VisDrone val images extracted above. Prints MB before->after (% smaller).
!python -m src.optimization.quantize_int8 \
    --model weights/best.onnx \
    --calib-dir /content/data/yolo/VisDrone-DET/images/val \
    --num-samples 200 --imgsz 640

In [ ]:
# --- (c) Build the TensorRT INT8 engine (GPU) -> weights/best.int8.engine ---
!python -m src.optimization.build_tensorrt \
    --onnx weights/best.onnx \
    --calib-dir /content/data/yolo/VisDrone-DET/images/val \
    --engine weights/best.int8.engine \
    --num-samples 200 --imgsz 640

In [ ]:
# --- (d) Benchmark FP32 vs ONNX-INT8 vs TensorRT-INT8 (size / latency / FPS) ---
# Run on the GPU box so the TensorRT row is measured, not skipped.
BENCH_MD = 'outputs/optimization_results.md'
!python -m src.optimization.benchmark --weights-dir weights --stem best --runs 100 --out {BENCH_MD}
print('\n--- benchmark table ---')
print(open(BENCH_MD).read())

In [ ]:
# --- Persist ALL outputs back to Drive (a disconnect loses nothing) ---
import os, shutil

DRIVE_WEIGHTS = f'{DRIVE_ROOT}/weights'
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)

artifacts = [
    'weights/best.onnx',
    'weights/best.int8.onnx',
    'weights/best.int8.engine',
    'outputs/optimization_results.md',
]
for a in artifacts:
    if os.path.exists(a):
        dest = os.path.join(DRIVE_WEIGHTS, os.path.basename(a))
        shutil.copy2(a, dest)
        print('Copied', a, '->', dest)
    else:
        print('SKIP (not produced):', a)

print('\nDone. Paste the benchmark table into the README Optimization section '
      '(report the measured numbers as-is).')